# Stream Processing Pipeline — Mercado de Criptomoedas
## MBA em Engenharia de Dados | Disciplina: Stream Processing Pipelines
**Prof. Rafael Tsuji Matsuyama** | Trabalho Final

---

### Visão Geral do Pipeline

Este notebook implementa um pipeline de dados de streaming que consome cotações de criptomoedas em tempo real da **CoinGecko API** (API pública, gratuita, sem autenticação), processa os dados com **Apache Spark Structured Streaming** no Databricks, e persiste o resultado em formato **Parquet** (Delta Lake).

```
CoinGecko API (JSON)
       │
       ▼
  Ingestão via polling (readStream)
       │
       ▼
  [BÔNUS 1] Validação de Schema + Filtro de qualidade
       │
       ▼
  [BÔNUS 2] Agregação: média, máx, mín por moeda
       │
       ▼
  [BÔNUS 3] Window Functions: janela deslizante de 1h
       │
       ▼
  Output: Delta Lake (Parquet) no DBFS
```

**Tecnologias utilizadas:**
- Apache Spark Structured Streaming (Databricks Runtime 13+)
- CoinGecko Public API v3
- Delta Lake (formato Parquet)
- Python 3.10+

**Fonte de dados:** [CoinGecko API](https://docs.coingecko.com) — maior agregador independente de dados de criptomoedas do mundo, com mais de 18.000 moedas e 1.000+ exchanges integradas. Tier gratuito: 30 chamadas/minuto, sem necessidade de API key.


---
## Célula 1 — Instalação de dependências e imports
Instalamos a biblioteca `requests` para as chamadas HTTP à API. No Databricks, o `pyspark` já está disponível no ambiente.

In [ ]:
# Instalação de dependências externas
# No Databricks, execute em um cluster com DBR 13.x ou superior
%pip install requests --quiet

# Imports padrão
import requests
import json
import time
import os
from datetime import datetime
from pathlib import Path

# Imports PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType, TimestampType
)
from pyspark.sql.window import Window

print("✅ Dependências carregadas com sucesso")

---
## Célula 2 — Configuração do ambiente Databricks e diretórios

Configuramos os caminhos no DBFS (Databricks File System) para:
- `landing_path`: onde os arquivos JSON brutos chegam (zona de pouso)
- `output_path`: onde o Parquet processado é salvo (zona de consumo)
- `checkpoint_path`: estado do stream (obrigatório para retomada exata em caso de falha)

In [ ]:
# ── Configuração do SparkSession ──────────────────────────────────────────────
# No Databricks, o SparkSession já existe como 'spark'.
# Este bloco garante compatibilidade com execução local também.

try:
    spark  # Databricks: variável já existe no contexto
    DATABRICKS = True
    print("✅ Executando no Databricks")
except NameError:
    spark = (
        SparkSession.builder
        .appName("StreamPipeline_CoinGecko")
        .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoint")
        .getOrCreate()
    )
    DATABRICKS = False
    print("✅ Executando localmente")

# ── Definição dos caminhos no DBFS ────────────────────────────────────────────
BASE_PATH       = "/dbfs/tmp/crypto_pipeline"   # raiz do projeto no DBFS
LANDING_PATH    = f"{BASE_PATH}/landing"         # JSONs brutos (zona de pouso)
OUTPUT_PATH     = f"{BASE_PATH}/output"          # Parquet processado
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoint"      # Estado do stream
AGGS_PATH       = f"{BASE_PATH}/aggregations"    # Resultados de agregação

# Cria os diretórios se não existirem
for path in [LANDING_PATH, OUTPUT_PATH, CHECKPOINT_PATH, AGGS_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"\n📁 Diretórios configurados:")
print(f"   Landing (JSON bruto):  {LANDING_PATH}")
print(f"   Output  (Parquet):     {OUTPUT_PATH}")
print(f"   Checkpoint (estado):   {CHECKPOINT_PATH}")
print(f"   Agregações:            {AGGS_PATH}")

---
## Célula 3 — [BÔNUS 1] Definição do Schema Estruturado

Antes de qualquer dado entrar no pipeline, definimos explicitamente o schema esperado. Isso é chamado de **schema-on-read** — o Spark valida cada evento contra esse esquema na entrada.

Campos esperados da CoinGecko API (`/coins/markets`):
- `id`: identificador único da moeda (ex: `bitcoin`)
- `symbol`: símbolo de mercado (ex: `btc`)
- `name`: nome completo (ex: `Bitcoin`)
- `current_price`: preço atual em USD
- `market_cap`: capitalização de mercado em USD
- `total_volume`: volume de negociação nas últimas 24h
- `price_change_percentage_24h`: variação percentual em 24h
- `last_updated`: timestamp da última atualização

In [ ]:
# ── [BÔNUS 1] Schema estruturado dos eventos de entrada ──────────────────────
# Definir schema explicitamente é uma boa prática em engenharia de dados:
# - Garante que campos obrigatórios estejam presentes
# - Evita inferência automática (cara em streaming)
# - Rejeita ou sinaliza eventos malformados

COIN_SCHEMA = StructType([
    StructField("id",                          StringType(),  nullable=False),  # obrigatório
    StructField("symbol",                      StringType(),  nullable=False),  # obrigatório
    StructField("name",                        StringType(),  nullable=False),  # obrigatório
    StructField("current_price",               DoubleType(),  nullable=True),
    StructField("market_cap",                  DoubleType(),  nullable=True),
    StructField("market_cap_rank",             LongType(),    nullable=True),
    StructField("total_volume",                DoubleType(),  nullable=True),
    StructField("high_24h",                    DoubleType(),  nullable=True),
    StructField("low_24h",                     DoubleType(),  nullable=True),
    StructField("price_change_percentage_24h", DoubleType(),  nullable=True),
    StructField("last_updated",                StringType(),  nullable=True),
])

# Campos obrigatórios — eventos sem estes campos serão descartados
REQUIRED_FIELDS = ["id", "symbol", "name", "current_price"]

print("✅ Schema definido com", len(COIN_SCHEMA.fields), "campos")
print("\nCampos obrigatórios:", REQUIRED_FIELDS)
print("\nSchema completo:")
for field in COIN_SCHEMA.fields:
    obrig = "[OBRIGATÓRIO]" if not field.nullable else "[opcional]    "
    print(f"  {obrig}  {field.name:<40} {str(field.dataType):<20}")

---
## Célula 4 — Função de ingestão: CoinGecko API → JSON landing

Implementamos o **produtor de dados**: uma função que chama a CoinGecko API e persiste cada resposta como um arquivo JSON no diretório de landing. O Spark Structured Streaming monitora esse diretório continuamente e processa novos arquivos à medida que chegam — este é o padrão **file-based streaming source**.

**Por que polling e não WebSocket?**
A CoinGecko API gratuita é REST/JSON — não oferece WebSocket nativo no tier free. O padrão de polling (pull a cada N segundos) é perfeitamente válido para streaming de dados de mercado, onde a janela de atualização é de 1 a 5 minutos. Para o trabalho, executaremos o produtor em um loop antes de iniciar o pipeline.

In [ ]:
# ── Configuração da CoinGecko API ─────────────────────────────────────────────

COINGECKO_URL = "https://api.coingecko.com/api/v3/coins/markets"

# Moedas monitoradas — top 10 por capitalização de mercado
COINS_TO_TRACK = [
    "bitcoin", "ethereum", "tether", "binancecoin", "solana",
    "ripple", "usd-coin", "staked-ether", "dogecoin", "cardano"
]

API_PARAMS = {
    "vs_currency": "usd",
    "ids": ",".join(COINS_TO_TRACK),
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": False,
    "price_change_percentage": "24h"
}

def fetch_coin_data() -> list[dict]:
    """
    Chama a CoinGecko API e retorna a lista de moedas com seus dados de mercado.
    Retorna lista vazia em caso de erro (pipeline continua sem interrupção).
    """
    try:
        headers = {"accept": "application/json"}
        response = requests.get(COINGECKO_URL, params=API_PARAMS, headers=headers, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        # 429 = rate limit atingido — aguardar e tentar novamente
        if e.response.status_code == 429:
            print(f"⚠️  Rate limit (429) — aguardando 60s...")
            time.sleep(60)
        else:
            print(f"❌ Erro HTTP {e.response.status_code}: {e}")
        return []
    except Exception as e:
        print(f"❌ Erro na chamada à API: {e}")
        return []


def ingest_batch(batch_num: int) -> str:
    """
    Busca dados da API, adiciona metadados de ingestão e salva no landing.
    Retorna o caminho do arquivo salvo.
    """
    coins = fetch_coin_data()
    if not coins:
        return None

    # Adiciona timestamp de ingestão a cada registro
    ingestion_ts = datetime.utcnow().isoformat()
    for coin in coins:
        coin["ingestion_timestamp"] = ingestion_ts
        # Garante que campos numéricos existam (validação mínima antes de salvar)
        coin["current_price"] = coin.get("current_price") or 0.0
        coin["market_cap"]    = coin.get("market_cap")    or 0.0
        coin["total_volume"]  = coin.get("total_volume")  or 0.0

    filename = f"{LANDING_PATH}/batch_{batch_num:04d}_{int(time.time())}.json"
    with open(filename, "w") as f:
        json.dump(coins, f)

    print(f"📥 Batch {batch_num:04d} salvo — {len(coins)} moedas — {ingestion_ts}")
    return filename


# ── Teste de conectividade com a API ──────────────────────────────────────────
print("🔌 Testando conexão com a CoinGecko API...")
test_data = fetch_coin_data()
if test_data:
    print(f"✅ API respondeu com {len(test_data)} moedas")
    print(f"\nExemplo de registro (bitcoin):")
    btc = next((c for c in test_data if c["id"] == "bitcoin"), test_data[0])
    print(f"  id:             {btc.get('id')}")
    print(f"  name:           {btc.get('name')}")
    print(f"  current_price:  USD {btc.get('current_price'):,.2f}")
    print(f"  market_cap:     USD {btc.get('market_cap'):,.0f}")
    print(f"  24h change:     {btc.get('price_change_percentage_24h'):+.2f}%")
    print(f"  last_updated:   {btc.get('last_updated')}")
else:
    print("❌ Falha na conexão com a API")

---
## Célula 5 — Geração de dados para o landing zone

Executamos o produtor em loop para simular a chegada contínua de dados. Cada iteração chama a API, adiciona timestamp de ingestão e salva um arquivo JSON no diretório de landing.

**Parâmetros:**
- `NUM_BATCHES = 5`: gera 5 snapshots com intervalo de 15 segundos entre cada um
- Total de eventos: ~50 registros (10 moedas × 5 batches)
- Tempo total: ~75 segundos

> **Nota para avaliação:** Caso queira mais dados para os bônus de janelamento, aumente `NUM_BATCHES` para 20 ou mais. O pipeline processa qualquer quantidade.

In [ ]:
# ── Produtor de dados: gera batches no landing zone ───────────────────────────
# Ajuste NUM_BATCHES conforme necessário:
#   5  batches → demonstração rápida (~75s)
#  20  batches → melhor visualização das janelas temporais (~5min)

NUM_BATCHES      = 5    # número de chamadas à API
POLL_INTERVAL    = 15   # segundos entre cada chamada (respeitando rate limit)

print(f"🚀 Iniciando ingestão — {NUM_BATCHES} batches com intervalo de {POLL_INTERVAL}s")
print(f"   Tempo total estimado: {NUM_BATCHES * POLL_INTERVAL}s\n")

files_created = []
for i in range(1, NUM_BATCHES + 1):
    filepath = ingest_batch(i)
    if filepath:
        files_created.append(filepath)
    if i < NUM_BATCHES:
        time.sleep(POLL_INTERVAL)

print(f"\n✅ Ingestão concluída — {len(files_created)} arquivos criados no landing zone")
print(f"📁 Diretório: {LANDING_PATH}")

---
## Célula 6 — [ENUNCIADO BASE] Pipeline Spark Structured Streaming

Esta é a célula central do trabalho. O Spark lê os arquivos JSON do landing zone via `readStream`, aplica o schema definido no Bônus 1, e escreve o resultado em Parquet via `writeStream`.

**Output Mode: `append`** — o mais eficiente para eventos novos: apenas os registros do micro-batch atual são gravados, sem precisar regerar toda a tabela.

**Trigger: `availableNow`** — processa todos os arquivos disponíveis agora e encerra. Ideal para demonstração. Em produção, usaríamos `processingTime='30 seconds'`.

In [ ]:
# ── [ENUNCIADO BASE] Leitura do stream de JSON ────────────────────────────────
# readStream com source 'json' monitora o diretório e processa novos arquivos
# O schema explícito evita inferência automática (cara em streaming)

# Schema completo incluindo campo de ingestão adicionado pelo produtor
FULL_SCHEMA = StructType(
    COIN_SCHEMA.fields + [
        StructField("ingestion_timestamp", StringType(), nullable=True)
    ]
)

raw_stream = (
    spark.readStream
    .format("json")
    .schema(FULL_SCHEMA)             # schema explícito (Bônus 1)
    .option("multiLine", True)       # cada arquivo contém um array JSON
    .option("mode", "DROPMALFORMED") # descarta registros malformados silenciosamente
    .load(LANDING_PATH)
)

print("✅ Stream de leitura configurado")
print(f"   Fonte:  {LANDING_PATH}")
print(f"   Formato: JSON (multiLine=True)")
print(f"   Modo de erros: DROPMALFORMED")
print(f"   Schema: {len(FULL_SCHEMA.fields)} campos")

In [ ]:
# ── [ENUNCIADO BASE] Transformação: JSON → Parquet ────────────────────────────
# Transformações aplicadas antes da escrita:
#   1. Conversão do campo last_updated (string ISO) para TimestampType
#   2. Conversão do campo ingestion_timestamp para TimestampType
#   3. Cálculo do spread (diferença entre high_24h e low_24h)
#   4. Classificação da variação diária (alta / queda / estável)

transformed_stream = (
    raw_stream
    # ── Bônus 1: filtrar registros sem preço ou com preço zerado ──
    .filter(
        F.col("current_price").isNotNull() &
        (F.col("current_price") > 0) &
        F.col("id").isNotNull()
    )
    # ── Transformações de tipo e enriquecimento ───────────────────
    .withColumn(
        "event_time",
        F.to_timestamp(F.col("last_updated"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
    )
    .withColumn(
        "ingestion_ts",
        F.to_timestamp(F.col("ingestion_timestamp"))
    )
    .withColumn(
        "spread_24h",
        F.round(F.col("high_24h") - F.col("low_24h"), 4)
    )
    .withColumn(
        "trend_24h",
        F.when(F.col("price_change_percentage_24h") >  1.0,  "alta")
         .when(F.col("price_change_percentage_24h") < -1.0,  "queda")
         .otherwise("estavel")
    )
    # ── Seleciona colunas finais para o Parquet ───────────────────
    .select(
        "id", "symbol", "name",
        "current_price", "market_cap", "market_cap_rank",
        "total_volume", "high_24h", "low_24h",
        "spread_24h",
        "price_change_percentage_24h",
        "trend_24h",
        "event_time",
        "ingestion_ts"
    )
)

print("✅ Transformações configuradas")
print("\nColunas no output final:")
for field in transformed_stream.schema.fields:
    print(f"  {field.name:<35} {str(field.dataType):<20}")

In [ ]:
# ── [ENUNCIADO BASE] Escrita do stream em Parquet (Delta Lake) ────────────────
# writeStream com:
#   - format: delta (Parquet com ACID transactions — padrão no Databricks)
#   - outputMode: append (apenas novos registros — mais eficiente)
#   - checkpointLocation: persiste estado do pipeline (permite retomada)
#   - trigger: availableNow → processa todos os arquivos disponíveis e para

stream_query = (
    transformed_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)      # para demonstração; em produção: processingTime='30 seconds'
    .start(OUTPUT_PATH)
)

# Aguarda o término do processamento
stream_query.awaitTermination()

print("\n✅ Stream processado com sucesso")
print(f"📦 Output salvo em: {OUTPUT_PATH} (formato Delta/Parquet)")

---
## Célula 7 — Verificação do output base

Lemos o Parquet resultante para confirmar que o pipeline funcionou corretamente e que os dados estão no formato esperado.

In [ ]:
# ── Leitura e verificação do output ──────────────────────────────────────────
output_df = spark.read.format("delta").load(OUTPUT_PATH)

total_records = output_df.count()
print(f"📊 Total de registros no output: {total_records}")
print(f"   Colunas: {len(output_df.columns)}")

print("\n📋 Amostra dos dados processados (5 registros):")
output_df.select(
    "symbol", "name", "current_price",
    "price_change_percentage_24h", "trend_24h", "event_time"
).show(5, truncate=False)

print("\n📈 Distribuição de tendências (24h):")
output_df.groupBy("trend_24h").count().orderBy("count", ascending=False).show()

---
## Célula 8 — [BÔNUS 2] Agregação dos dados de streaming

Aplicamos operações de agregação sobre o dataset processado:
- `avg()`: preço médio observado durante o período de monitoramento
- `max()`: maior preço registrado (high)
- `min()`: menor preço registrado (low)
- `count()`: número de snapshots capturados
- `last()`: último preço registrado (mais recente)

Agrupamos por `symbol` (identificador da moeda) para ter uma visão consolidada por ativo.

In [ ]:
# ── [BÔNUS 2] Agregações por moeda ────────────────────────────────────────────

aggregated_df = (
    output_df
    .groupBy("id", "symbol", "name")
    .agg(
        F.count("*")                                          .alias("snapshots"),
        F.avg("current_price")                               .alias("avg_price_usd"),
        F.max("current_price")                               .alias("max_price_usd"),
        F.min("current_price")                               .alias("min_price_usd"),
        F.last("current_price")                              .alias("last_price_usd"),
        F.avg("price_change_percentage_24h")                 .alias("avg_change_24h_pct"),
        F.avg("total_volume")                                .alias("avg_volume_usd"),
        F.avg("market_cap")                                  .alias("avg_mktcap_usd"),
        F.max("ingestion_ts")                                .alias("last_seen"),
        # Percentual de snapshots em alta vs queda
        F.round(
            F.sum(F.when(F.col("trend_24h") == "alta",  1).otherwise(0)) /
            F.count("*") * 100, 1
        ).alias("pct_snapshots_alta")
    )
    .withColumn("avg_price_usd",     F.round("avg_price_usd",     4))
    .withColumn("max_price_usd",     F.round("max_price_usd",     4))
    .withColumn("min_price_usd",     F.round("min_price_usd",     4))
    .withColumn("last_price_usd",    F.round("last_price_usd",    4))
    .withColumn("avg_change_24h_pct",F.round("avg_change_24h_pct",2))
    .orderBy("avg_mktcap_usd", ascending=False)
)

print("📊 [BÔNUS 2] Agregações por moeda — período monitorado:")
aggregated_df.select(
    "symbol", "name", "snapshots",
    "avg_price_usd", "max_price_usd", "min_price_usd",
    "avg_change_24h_pct"
).show(10, truncate=False)

# Persiste as agregações em Parquet
aggregated_df.write.mode("overwrite").format("delta").save(AGGS_PATH)
print(f"\n✅ Agregações salvas em: {AGGS_PATH}")

---
## Célula 9 — [BÔNUS 3] Window Functions — Janela deslizante (Sliding Window)

Implementamos **janelas temporais** para calcular métricas dentro de uma janela deslizante:

- **Janela**: 60 minutos de duração
- **Slide**: avança a cada 5 minutos
- **Resultado**: para cada moeda e cada janela de 60min, calculamos preço médio, máximo, mínimo e volatilidade

Este é o padrão clássico para monitoramento financeiro: "preço médio nas últimas 1h, atualizado a cada 5 minutos".

Também implementamos uma **janela fixa (tumbling window)** para comparação — ela divide o tempo em blocos estanques sem sobreposição.

In [ ]:
# ── [BÔNUS 3] Janela deslizante (Sliding Window) ──────────────────────────────
# Calcula métricas dentro de uma janela de 60min que avança a cada 5min
# Padrão clássico para trading: "últimas 1h de cotação, atualizado a cada 5min"

sliding_window_df = (
    output_df
    .groupBy(
        "symbol",
        "name",
        # F.window(event_time, windowDuration, slideDuration)
        # windowDuration: duração total da janela = 60 minutos
        # slideDuration:  de quanto em quanto a janela avança = 5 minutos
        F.window(F.col("event_time"), "60 minutes", "5 minutes")
    )
    .agg(
        F.count("*")                   .alias("eventos_na_janela"),
        F.avg("current_price")         .alias("preco_medio_usd"),
        F.max("current_price")         .alias("preco_max_usd"),
        F.min("current_price")         .alias("preco_min_usd"),
        # Volatilidade aproximada: (max - min) / avg * 100
        F.round(
            (F.max("current_price") - F.min("current_price")) /
            F.avg("current_price") * 100, 4
        ).alias("volatilidade_pct"),
        F.avg("total_volume")          .alias("volume_medio_usd")
    )
    .withColumn("preco_medio_usd", F.round("preco_medio_usd", 4))
    .withColumn("preco_max_usd",   F.round("preco_max_usd",   4))
    .withColumn("preco_min_usd",   F.round("preco_min_usd",   4))
    .withColumn("window_start",    F.col("window.start"))
    .withColumn("window_end",      F.col("window.end"))
    .drop("window")
    .orderBy("symbol", "window_start")
)

print("⏱️  [BÔNUS 3] Janela deslizante — 60min, slide de 5min:")
sliding_window_df.select(
    "symbol", "window_start", "window_end",
    "eventos_na_janela", "preco_medio_usd",
    "preco_max_usd", "preco_min_usd", "volatilidade_pct"
).show(20, truncate=False)

print(f"\nTotal de janelas calculadas: {sliding_window_df.count()}")

In [ ]:
# ── [BÔNUS 3] Janela fixa (Tumbling Window) ────────────────────────────────────
# Divide o tempo em blocos estanques de 30 minutos sem sobreposição
# Cada evento pertence a exatamente UMA janela

tumbling_window_df = (
    output_df
    .groupBy(
        "symbol",
        "name",
        # F.window com apenas windowDuration = tumbling window
        F.window(F.col("event_time"), "30 minutes")
    )
    .agg(
        F.count("*")          .alias("eventos"),
        F.avg("current_price").alias("preco_medio_usd"),
        F.max("current_price").alias("preco_abertura_est"),
        F.min("current_price").alias("preco_fechamento_est"),
        F.avg("total_volume") .alias("volume_medio_usd")
    )
    .withColumn("preco_medio_usd",        F.round("preco_medio_usd",        4))
    .withColumn("preco_abertura_est",     F.round("preco_abertura_est",     4))
    .withColumn("preco_fechamento_est",   F.round("preco_fechamento_est",   4))
    .withColumn("janela_inicio",          F.col("window.start"))
    .withColumn("janela_fim",             F.col("window.end"))
    .drop("window")
    .orderBy("symbol", "janela_inicio")
)

print("⏱️  [BÔNUS 3] Janela fixa (tumbling) — blocos de 30min:")
tumbling_window_df.show(20, truncate=False)

# Persiste resultados das janelas
WINDOWS_PATH = f"{BASE_PATH}/windows"
sliding_window_df.write.mode("overwrite").format("delta").save(f"{WINDOWS_PATH}/sliding")
tumbling_window_df.write.mode("overwrite").format("delta").save(f"{WINDOWS_PATH}/tumbling")
print(f"\n✅ Janelas salvas em: {WINDOWS_PATH}")

---
## Célula 10 — Resumo executivo do pipeline

Consolida todas as métricas do pipeline em um relatório final, evidenciando o que foi entregue em cada etapa.

In [ ]:
# ── Resumo executivo do pipeline ──────────────────────────────────────────────
print("=" * 70)
print("  RESUMO DO PIPELINE — Stream Processing | MBA Engenharia de Dados")
print("=" * 70)

print("\n📥 INGESTÃO (Enunciado Base)")
print(f"   Fonte:          CoinGecko Public API v3")
print(f"   Moedas:         {', '.join(COINS_TO_TRACK[:5])} + mais 5")
print(f"   Batches:        {NUM_BATCHES} chamadas × {len(COINS_TO_TRACK)} moedas")
print(f"   Total eventos:  {total_records} registros")
print(f"   Formato input:  JSON (array por arquivo)")
print(f"   Formato output: Delta Lake (Parquet) com ACID")

print("\n✅ [BÔNUS 1] VALIDAÇÃO / SCHEMA ESTRUTURADO")
print(f"   Schema com {len(FULL_SCHEMA.fields)} campos definidos explicitamente")
print(f"   Campos obrigatórios validados: {', '.join(REQUIRED_FIELDS)}")
print(f"   Registros com price <= 0 filtrados")
print(f"   Modo de erro: DROPMALFORMED (rejeita JSON inválido)")

print("\n✅ [BÔNUS 2] AGREGAÇÃO")
aggs_count = aggregated_df.count()
print(f"   Moedas agregadas: {aggs_count}")
print(f"   Métricas: avg_price, max_price, min_price, avg_volume, avg_change_24h")
print(f"   Classificação de tendência: alta / queda / estável")

print("\n✅ [BÔNUS 3] WINDOW FUNCTIONS")
sw_count = sliding_window_df.count()
tw_count = tumbling_window_df.count()
print(f"   Sliding window:  60min × slide 5min  → {sw_count} janelas calculadas")
print(f"   Tumbling window: blocos de 30min      → {tw_count} janelas calculadas")
print(f"   Métrica-chave:   volatilidade_pct por janela")

print("\n📁 OUTPUTS GERADOS")
print(f"   {OUTPUT_PATH}          ← dados transformados (Parquet)")
print(f"   {AGGS_PATH}     ← agregações por moeda (Parquet)")
print(f"   {WINDOWS_PATH}/sliding ← janela deslizante (Parquet)")
print(f"   {WINDOWS_PATH}/tumbling← janela fixa (Parquet)")
print(f"   {CHECKPOINT_PATH}   ← estado do stream (retomada)")

print("\n" + "=" * 70)
print("  Pipeline concluído com sucesso.")
print("=" * 70)

---
## Apêndice — [BÔNUS 4] Configuração de Deploy / CI-CD

O Bônus 4 exige evidenciar um processo de deploy estruturado do pipeline. As configurações abaixo representam a camada de infraestrutura do trabalho.

### Estratégia de Deploy no Databricks

Em produção, o pipeline seria configurado como um **Databricks Job** executado periodicamente. O arquivo `databricks.yml` abaixo configura o deploy via Databricks Asset Bundles (DAB) — a abordagem moderna de "pipeline como código" no ecossistema Databricks.

```yaml
# databricks.yml — Configuração do pipeline como código (Databricks Asset Bundle)
bundle:
  name: stream_pipeline_coingecko

resources:
  jobs:
    crypto_stream_job:
      name: "CryptoStream - MBA Pipeline"
      description: "Pipeline de streaming de cotações de criptomoedas (CoinGecko API)"

      # Agendamento: executa a cada 15 minutos em produção
      schedule:
        quartz_cron_expression: "0 */15 * * * ?"
        timezone_id: "America/Sao_Paulo"

      # Configuração do cluster
      job_clusters:
        - job_cluster_key: streaming_cluster
          new_cluster:
            spark_version: "13.3.x-scala2.12"
            node_type_id: "Standard_DS3_v2"
            num_workers: 2
            spark_conf:
              spark.sql.streaming.checkpointLocation: "/dbfs/tmp/crypto_pipeline/checkpoint"
              spark.databricks.delta.autoCompact.enabled: "true"

      # Tarefas (Tasks) do job
      tasks:
        - task_key: ingest
          notebook_task:
            notebook_path: "notebooks/stream_pipeline_coingecko"
            base_parameters:
              NUM_BATCHES: "5"
              POLL_INTERVAL: "15"
          job_cluster_key: streaming_cluster
          libraries:
            - pypi:
                package: requests

      # Notificações em caso de falha
      email_notifications:
        on_failure:
          - engenharia@empresa.com.br
```

### GitHub Actions — CI/CD do Pipeline

```yaml
# .github/workflows/deploy_pipeline.yml
name: Deploy Stream Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  validate:
    name: Validar notebook
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3

      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Instalar dependências de validação
        run: |
          pip install nbformat nbconvert requests

      - name: Validar sintaxe do notebook
        run: |
          python -c "
          import nbformat
          nb = nbformat.read('notebooks/stream_pipeline_coingecko.ipynb', as_version=4)
          print(f'Notebook válido: {len(nb.cells)} células')
          "

      - name: Verificar conectividade com CoinGecko API
        run: |
          python -c "
          import requests
          r = requests.get('https://api.coingecko.com/api/v3/ping', timeout=10)
          assert r.status_code == 200, f'API indisponível: {r.status_code}'
          print('API disponível:', r.json())
          "

  deploy:
    name: Deploy no Databricks
    needs: validate
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v3

      - name: Instalar Databricks CLI
        run: pip install databricks-cli

      - name: Deploy via Databricks Asset Bundle
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
        run: |
          databricks bundle deploy --target prod
          echo "✅ Pipeline deployado em produção"
```

> **Nota:** O hash de integridade gerado automaticamente em cada execução da esteira (SHA do commit + timestamp) permite rastrear qualquer adulteração nos logs de execução, criando a trilha de auditoria mencionada pelo professor na Aula 04.